# Geophone SNN Classifier
3-class classification (human / car / nothing) using a Spiking Neural Network on EG-4.5-II geophone data.

**Architecture:** Input(32) → LIF(128) → LIF(64) → LIF(3, integrator)

**Detection latency:** ~1.0–1.5s from first footstep to detection output

In [ ]:
# Cell 1: Configuration & Imports
import subprocess, sys
for pkg in ['spikingjelly', 'scipy', 'scikit-learn', 'matplotlib']:
    try: __import__(pkg.replace('-','_').split('-')[0])
    except ImportError: subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import torch
import torch.nn as nn
from spikingjelly.activation_based import neuron, surrogate, layer, functional as sj_functional
import numpy as np
import sqlite3
import os
import time
import copy
from scipy import signal as sig
from scipy import stats as spstats
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

# Resolve project directory so paths work on any machine. Jupyter has no __file__,
# so probe likely locations and pick the first that actually contains the DB.
def _find_project_dir():
    candidates = [
        os.getcwd(),
        os.path.abspath('.'),
        r'S:\ALL PROJECTS\geophone sensor\finals project\finals project',
    ]
    for d in candidates:
        if os.path.exists(os.path.join(d, 'geophone_data.db')):
            return d
    return candidates[0]

PROJECT_DIR = _find_project_dir()

CONFIG = {
    'SAMPLE_RATE': 1000,
    'WINDOW_SIZE': 1000,
    'STRIDE': 500,
    'NUM_FEATURES': 32,
    'TIME_STEPS': 25,
    'BATCH_SIZE': 16,
    'EPOCHS': 75,
    'LR': 1e-3,
    'WEIGHT_DECAY': 1e-4,
    'EMA_DECAY': 0.9997,
    'DROPOUT': 0.1,
    'SPIKE_REG_LAMBDA': 0.001,
    'SPIKE_REG_TARGET': 0.1,
    'COSINE_ETA_MIN': 1e-6,
    'CLIP_RANGE': 5.0,
    'HYSTERESIS_ENTRY': 0.7,
    'HYSTERESIS_EXIT': 0.35,
    'DB_PATH': os.path.join(PROJECT_DIR, 'geophone_data.db'),
    'DEVICE': 'cpu',
}

CLASS_NAMES = ['human', 'car', 'nothing']
print(f'Device: {CONFIG["DEVICE"]}')
print(f'PyTorch: {torch.__version__}')
print(f'Project dir: {PROJECT_DIR}')
print(f'DB path: {CONFIG["DB_PATH"]}')
print(f'DB exists: {os.path.exists(CONFIG["DB_PATH"])}')

In [ ]:
# Cell 2: Data Loading
def load_table(db_path, table_name):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    cursor.execute(f'SELECT amplitude FROM "{table_name}" ORDER BY time_s')
    data = np.array([r[0] for r in cursor.fetchall()], dtype=np.float32)
    conn.close()
    return data

raw_data = {}
for table in ['human', 'human_nothing', 'car', 'car_nothing']:
    raw_data[table] = load_table(CONFIG['DB_PATH'], table)
    dur = len(raw_data[table]) / CONFIG['SAMPLE_RATE']
    print(f'{table}: {len(raw_data[table]):,} samples ({dur:.1f}s)')

In [ ]:
# Cell 3: Temporal Train/Val Split
def temporal_split(data, train_ratio=0.8):
    idx = int(len(data) * train_ratio)
    return data[:idx], data[idx:]

split_data = {}
for table in raw_data:
    train, val = temporal_split(raw_data[table])
    split_data[table] = {'train': train, 'val': val}
    t_dur = len(train) / CONFIG['SAMPLE_RATE']
    v_dur = len(val) / CONFIG['SAMPLE_RATE']
    print(f'{table}: train={len(train):,} ({t_dur:.1f}s) | val={len(val):,} ({v_dur:.1f}s)')

In [ ]:
# Cell 4: Windowing
def create_windows(data, window_size, stride):
    n = (len(data) - window_size) // stride + 1
    if n <= 0:
        return np.empty((0, window_size), dtype=np.float32)
    windows = np.array([data[i*stride : i*stride + window_size] for i in range(n)])
    return windows

def build_windowed_dataset(split_data, split_name):
    ws = CONFIG['WINDOW_SIZE']
    stride = CONFIG['STRIDE']
    
    human_w = create_windows(split_data['human'][split_name], ws, stride)
    car_w = create_windows(split_data['car'][split_name], ws, stride)
    nothing_h = create_windows(split_data['human_nothing'][split_name], ws, stride)
    nothing_c = create_windows(split_data['car_nothing'][split_name], ws, stride)
    nothing_w = np.concatenate([nothing_h, nothing_c], axis=0)
    
    windows = np.concatenate([human_w, car_w, nothing_w], axis=0)
    labels = np.concatenate([
        np.zeros(len(human_w), dtype=np.int64),
        np.ones(len(car_w), dtype=np.int64),
        np.full(len(nothing_w), 2, dtype=np.int64),
    ])
    print(f'  {split_name}: human={len(human_w)}, car={len(car_w)}, nothing={len(nothing_w)}, total={len(windows)}')
    return windows, labels

print('Building windowed datasets:')
train_windows, train_labels = build_windowed_dataset(split_data, 'train')
val_windows, val_labels = build_windowed_dataset(split_data, 'val')

In [ ]:
# Cell 5: Feature Extraction (32 features)
BANDS = {
    'LOW_FREQ': (20, 30),
    'CAR_APPROACH': (30, 34),
    'CAR_PEAK': (34, 40),
    'CAR_TAIL': (40, 48),
    'MID_GAP': (48, 60),
    'HUMAN_PEAK': (60, 70),
    'HUMAN_TAIL': (70, 80),
    'HIGH_FREQ': (90, 100),
}

FEATURE_NAMES = [
    'energy_low_freq', 'energy_car_approach', 'energy_car_peak', 'energy_car_tail',
    'energy_mid_gap', 'energy_human_peak', 'energy_human_tail', 'energy_high_freq',
    'rms_total', 'peak_to_peak', 'variance', 'zcr', 'kurtosis', 'skewness',
    'spectral_centroid', 'spectral_bandwidth', 'spectral_rolloff', 'spectral_flatness',
    'spectral_entropy', 'dominant_freq',
    'sta_lta_ratio', 'event_count',
    'mean_burst_len', 'activity_concentration', 'temporal_entropy', 'burst_efficiency',
    'autocorr_250', 'autocorr_500',
    'ratio_car_human', 'ratio_human_car',
    'centroid_car_band', 'centroid_human_band',
]

def design_bandpass(lowcut, highcut, fs, order=4):
    nyq = fs / 2
    low = max(lowcut / nyq, 0.001)
    high = min(highcut / nyq, 0.999)
    return sig.butter(order, [low, high], btype='band', output='sos')

bp_filters = {name: design_bandpass(lo, hi, CONFIG['SAMPLE_RATE']) for name, (lo, hi) in BANDS.items()}

def extract_features(window, fs=1000):
    feats = []
    
    # 1-8: Sub-band energies
    band_e = {}
    for name, sos in bp_filters.items():
        filtered = sig.sosfilt(sos, window)
        rms = np.sqrt(np.mean(filtered**2))
        band_e[name] = rms
        feats.append(rms)
    
    # 9: Total RMS
    rms_total = np.sqrt(np.mean(window**2))
    feats.append(rms_total)
    
    # 10: Peak-to-peak
    feats.append(np.ptp(window))
    
    # 11: Variance
    feats.append(np.var(window))
    
    # 12: Zero-crossing rate
    feats.append(np.sum(np.diff(np.sign(window)) != 0) / len(window))
    
    # 13: Kurtosis
    feats.append(float(spstats.kurtosis(window)))
    
    # 14: Skewness
    feats.append(float(spstats.skew(window)))
    
    # Spectral analysis
    fft_mag = np.abs(np.fft.rfft(window))
    freqs = np.fft.rfftfreq(len(window), d=1.0/fs)
    total_mag = np.sum(fft_mag) + 1e-10
    psd = fft_mag**2
    total_psd = np.sum(psd) + 1e-10
    
    # 15: Spectral centroid
    sc = np.sum(freqs * fft_mag) / total_mag
    feats.append(sc)
    
    # 16: Spectral bandwidth
    feats.append(np.sqrt(np.sum(((freqs - sc)**2) * fft_mag) / total_mag))
    
    # 17: Spectral rolloff (95%)
    cumsum = np.cumsum(psd)
    ro_idx = min(np.searchsorted(cumsum, 0.95 * total_psd), len(freqs) - 1)
    feats.append(freqs[ro_idx])
    
    # 18: Spectral flatness
    log_mean = np.mean(np.log(psd + 1e-10))
    feats.append(np.exp(log_mean) / (np.mean(psd) + 1e-10))
    
    # 19: Spectral entropy
    psd_norm = psd / total_psd
    feats.append(-np.sum(psd_norm * np.log2(psd_norm + 1e-10)))
    
    # 20: Dominant frequency
    feats.append(freqs[np.argmax(fft_mag)])
    
    # 21: STA/LTA ratio (max ratio using sliding 100ms STA)
    abs_w = np.abs(window)
    lta = np.mean(abs_w) + 1e-10
    sta_len = 100
    sta = np.convolve(abs_w, np.ones(sta_len)/sta_len, mode='valid')
    feats.append(np.max(sta) / lta)
    
    # 22: Event count
    peaks, _ = sig.find_peaks(abs_w, height=2*rms_total)
    feats.append(float(len(peaks)))
    
    # 23: Mean burst length
    above = abs_w > rms_total
    diff = np.diff(above.astype(np.int8))
    starts = np.where(diff == 1)[0]
    ends = np.where(diff == -1)[0]
    if len(starts) > 0 and len(ends) > 0:
        if ends[0] < starts[0]:
            ends = ends[1:]
        n_bursts = min(len(starts), len(ends))
        feats.append(float(np.mean(ends[:n_bursts] - starts[:n_bursts])) if n_bursts > 0 else 0.0)
    else:
        feats.append(0.0)
    
    # 24: Activity concentration
    sorted_e = np.sort(window**2)[::-1]
    q = len(sorted_e) // 4
    feats.append(np.sum(sorted_e[:q]) / (np.sum(sorted_e) + 1e-10))
    
    # 25: Temporal entropy
    hist, _ = np.histogram(window, bins=50, density=True)
    hist = hist[hist > 0]
    hist_norm = hist / (np.sum(hist) + 1e-10)
    feats.append(-np.sum(hist_norm * np.log2(hist_norm + 1e-10)))
    
    # 26: Burst efficiency
    burst_mask = abs_w > rms_total
    feats.append(np.sum(window[burst_mask]**2) / (np.sum(window**2) + 1e-10))
    
    # 27-28: Autocorrelation
    for lag in [250, 500]:
        if len(window) > lag:
            ac = np.corrcoef(window[:-lag], window[lag:])[0, 1]
            feats.append(ac if np.isfinite(ac) else 0.0)
        else:
            feats.append(0.0)
    
    # 29-30: Cross-band ratios
    car_e = band_e['CAR_PEAK'] + 1e-10
    human_e = band_e['HUMAN_PEAK'] + 1e-10
    feats.append(car_e / human_e)
    feats.append(human_e / car_e)
    
    # 31: Spectral centroid of car band (34-48 Hz)
    car_mask = (freqs >= 34) & (freqs <= 48)
    cm = fft_mag[car_mask]
    cf = freqs[car_mask]
    feats.append(np.sum(cf * cm) / (np.sum(cm) + 1e-10) if len(cm) > 0 else 0.0)
    
    # 32: Spectral centroid of human band (60-80 Hz)
    hm_mask = (freqs >= 60) & (freqs <= 80)
    hm = fft_mag[hm_mask]
    hf = freqs[hm_mask]
    feats.append(np.sum(hf * hm) / (np.sum(hm) + 1e-10) if len(hm) > 0 else 0.0)
    
    return np.array(feats, dtype=np.float32)

print(f'Extracting {CONFIG["NUM_FEATURES"]} features per window...')
t0 = time.time()
train_features = np.array([extract_features(w) for w in train_windows])
val_features = np.array([extract_features(w) for w in val_windows])
print(f'Done in {time.time()-t0:.1f}s')
print(f'Train: {train_features.shape}, Val: {val_features.shape}')

nan_count = np.sum(~np.isfinite(train_features))
if nan_count > 0:
    print(f'Warning: {nan_count} non-finite values, replacing with 0')
    train_features = np.nan_to_num(train_features, nan=0.0, posinf=0.0, neginf=0.0)
    val_features = np.nan_to_num(val_features, nan=0.0, posinf=0.0, neginf=0.0)

In [ ]:
# Cell 6: Normalization
train_mean = train_features.mean(axis=0)
train_std = train_features.std(axis=0) + 1e-8

train_normed = np.clip((train_features - train_mean) / train_std, -CONFIG['CLIP_RANGE'], CONFIG['CLIP_RANGE'])
val_normed = np.clip((val_features - train_mean) / train_std, -CONFIG['CLIP_RANGE'], CONFIG['CLIP_RANGE'])

print('Per-feature stats after normalization (train):')
print(f'  Mean range: [{train_normed.mean(axis=0).min():.4f}, {train_normed.mean(axis=0).max():.4f}]')
print(f'  Std range:  [{train_normed.std(axis=0).min():.4f}, {train_normed.std(axis=0).max():.4f}]')
print(f'  Clipped values: {np.sum(np.abs(train_normed) == CONFIG["CLIP_RANGE"])}')

In [ ]:
# Cell 7: Splice Augmentation (pre-computed)
def create_splice_samples(split_data, nothing_windows, n_splices=1000):
    """Insert positive signal bursts into nothing windows, then extract features."""
    human_train = split_data['human']['train']
    car_train = split_data['car']['train']
    
    spliced_features = []
    spliced_labels = []
    ws = CONFIG['WINDOW_SIZE']
    rng = np.random.default_rng(42)
    
    nothing_idx = np.where(train_labels == 2)[0]
    
    for i in range(n_splices):
        # Pick a random nothing window
        base_idx = rng.choice(nothing_idx)
        base = train_windows[base_idx].copy()
        
        # Pick class and source signal
        if rng.random() < 0.5:
            source = human_train
            label = 0
        else:
            source = car_train
            label = 1
        
        # Extract a random burst (200-600ms)
        burst_len = rng.integers(200, 601)
        max_start = len(source) - burst_len
        if max_start <= 0:
            continue
        src_start = rng.integers(0, max_start)
        burst = source[src_start : src_start + burst_len].copy()
        
        # Scale burst amplitude (simulate distance variation)
        burst *= rng.uniform(0.5, 1.5)
        
        # Insert position in base window
        max_insert = ws - burst_len
        if max_insert <= 0:
            continue
        insert_pos = rng.integers(0, max_insert)
        
        # Cosine crossfade (30 samples = 30ms)
        fade_len = min(30, burst_len // 4)
        fade_in = 0.5 * (1 - np.cos(np.pi * np.arange(fade_len) / fade_len))
        fade_out = fade_in[::-1]
        burst[:fade_len] *= fade_in
        burst[-fade_len:] *= fade_out
        
        # Mix into base
        spliced = base.copy()
        spliced[insert_pos : insert_pos + burst_len] += burst
        
        spliced_features.append(extract_features(spliced))
        spliced_labels.append(label)
    
    return np.array(spliced_features, dtype=np.float32), np.array(spliced_labels, dtype=np.int64)

print('Creating splice-augmented samples...')
t0 = time.time()
splice_feats, splice_labels = create_splice_samples(split_data, train_windows, n_splices=1500)
print(f'Created {len(splice_feats)} spliced samples in {time.time()-t0:.1f}s')

# Normalize spliced features with training statistics
splice_normed = np.clip((splice_feats - train_mean) / train_std, -CONFIG['CLIP_RANGE'], CONFIG['CLIP_RANGE'])

# Combine with training set
train_all_features = np.concatenate([train_normed, splice_normed], axis=0)
train_all_labels = np.concatenate([train_labels, splice_labels], axis=0)

print(f'\nFinal training set: {len(train_all_features)} samples')
for c in range(3):
    print(f'  {CLASS_NAMES[c]}: {np.sum(train_all_labels == c)}')

In [ ]:
# Cell 8: PyTorch Datasets & DataLoaders
class GeoDataset(Dataset):
    def __init__(self, features, labels, augment=False, feature_std=None):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.augment = augment
        self.feature_std = feature_std
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        x = self.features[idx].clone()
        y = self.labels[idx]
        
        if self.augment:
            # Gaussian jitter
            if torch.rand(1).item() < 0.5:
                noise = torch.randn_like(x) * 0.03
                x = x + noise
            
            # Amplitude scaling
            if torch.rand(1).item() < 0.5:
                scale = 0.8 + torch.rand(1).item() * 0.4
                x = x * scale
        
        return x, y

train_dataset = GeoDataset(train_all_features, train_all_labels, augment=True)
val_dataset = GeoDataset(val_normed, val_labels, augment=False)

train_loader = DataLoader(train_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=True, drop_last=False)
val_loader = DataLoader(val_dataset, batch_size=CONFIG['BATCH_SIZE'], shuffle=False)

steps_per_epoch = len(train_loader)
print(f'Train batches/epoch: {steps_per_epoch}')
print(f'Val batches: {len(val_loader)}')
print(f'EMA half-life: {-np.log(2)/np.log(CONFIG["EMA_DECAY"]):.0f} steps ({-np.log(2)/np.log(CONFIG["EMA_DECAY"])/steps_per_epoch:.1f} epochs)')

In [ ]:
# Cell 9: Model Definition (SpikingJelly)
class GeophoneSNN(nn.Module):
    def __init__(self, n_features=32, n_hidden1=128, n_hidden2=64, n_classes=3,
                 init_tau=2.0, v_threshold=0.5, time_steps=25, dropout=0.1):
        super().__init__()
        self.time_steps = time_steps
        sg = surrogate.Sigmoid(alpha=25.0)

        self.fc1 = layer.Linear(n_features, n_hidden1)
        self.lif1 = neuron.ParametricLIFNode(
            init_tau=init_tau, v_threshold=v_threshold,
            surrogate_function=sg, detach_reset=False)
        self.drop1 = nn.Dropout(dropout)

        self.fc2 = layer.Linear(n_hidden1, n_hidden2)
        self.lif2 = neuron.ParametricLIFNode(
            init_tau=init_tau, v_threshold=v_threshold,
            surrogate_function=sg, detach_reset=False)
        self.drop2 = nn.Dropout(dropout)

        self.fc3 = layer.Linear(n_hidden2, n_classes)
        self.lif3 = neuron.ParametricLIFNode(
            init_tau=init_tau, v_threshold=float('inf'),
            surrogate_function=sg, detach_reset=False)

    def forward(self, x):
        sj_functional.reset_net(self)

        spk1_rec, spk2_rec, mem3_rec = [], [], []

        for _ in range(self.time_steps):
            spk1 = self.lif1(self.fc1(x))
            spk1 = self.drop1(spk1)
            spk1_rec.append(spk1)

            spk2 = self.lif2(self.fc2(spk1))
            spk2 = self.drop2(spk2)
            spk2_rec.append(spk2)

            self.lif3(self.fc3(spk2))
            mem3_rec.append(self.lif3.v.clone())

        return torch.stack(spk1_rec), torch.stack(spk2_rec), torch.stack(mem3_rec)

model = GeophoneSNN(
    n_features=CONFIG['NUM_FEATURES'],
    time_steps=CONFIG['TIME_STEPS'],
    dropout=CONFIG['DROPOUT'],
).to(CONFIG['DEVICE'])

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Architecture: Input({CONFIG["NUM_FEATURES"]}) -> PLIF(128) -> PLIF(64) -> PLIF(3, integrator)')
print(f'Framework: SpikingJelly (ParametricLIFNode, learnable tau)')
print(f'init_tau=2.0, v_threshold=0.5 (tau=10 caused dead neurons)')

In [ ]:
# Cell 10: EMA Helper
class EMA:
    def __init__(self, model, decay):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()
    
    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name].mul_(self.decay).add_(param.data, alpha=1 - self.decay)
    
    def apply(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])
    
    def restore(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.backup[name])
        self.backup = {}

ema = EMA(model, CONFIG['EMA_DECAY'])
print(f'EMA initialized with decay={CONFIG["EMA_DECAY"]}')

In [ ]:
# Cell 11: Training Loop
from sklearn.metrics import f1_score

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['LR'], weight_decay=CONFIG['WEIGHT_DECAY'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['EPOCHS'], eta_min=CONFIG['COSINE_ETA_MIN'])
loss_fn = nn.CrossEntropyLoss()

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [],
           'train_f1': [], 'val_f1': [],
           'lr': [], 'firing_rate_l1': [], 'firing_rate_l2': []}
best_val_f1 = 0.0
best_val_acc = 0.0
best_model_state = None

print(f'Training for {CONFIG["EPOCHS"]} epochs, {steps_per_epoch} steps/epoch')
print(f'LR: {CONFIG["LR"]} -> {CONFIG["COSINE_ETA_MIN"]} (cosine)')
print(f'FR = avg firing rate of hidden neurons (target: {CONFIG["SPIKE_REG_TARGET"]})')
print('-' * 90)

t_start = time.time()

for epoch in range(CONFIG['EPOCHS']):
    # --- Train ---
    model.train()
    epoch_loss, epoch_correct, epoch_total = 0.0, 0, 0
    epoch_fr1, epoch_fr2 = 0.0, 0.0
    train_preds_all, train_labels_all = [], []

    for batch_x, batch_y in train_loader:
        batch_x = batch_x.to(CONFIG['DEVICE'])
        batch_y = batch_y.to(CONFIG['DEVICE'])

        spk1_rec, spk2_rec, mem3_rec = model(batch_x)

        # Vectorized loss: stack membrane potentials [T, B, C] -> [T*B, C]
        T, B, C = mem3_rec.shape
        mem_flat = mem3_rec.reshape(T * B, C)
        targets_flat = batch_y.unsqueeze(0).expand(T, -1).reshape(T * B)
        loss = loss_fn(mem_flat, targets_flat)

        # Spike rate regularization
        avg_fr1 = spk1_rec.mean()
        avg_fr2 = spk2_rec.mean()
        spike_reg = CONFIG['SPIKE_REG_LAMBDA'] * (
            (avg_fr1 - CONFIG['SPIKE_REG_TARGET'])**2 +
            (avg_fr2 - CONFIG['SPIKE_REG_TARGET'])**2
        )
        total_loss = loss + spike_reg

        optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ema.update(model)

        preds = mem3_rec[-1].argmax(dim=1)
        epoch_correct += (preds == batch_y).sum().item()
        epoch_total += len(batch_y)
        epoch_loss += total_loss.item()
        epoch_fr1 += avg_fr1.item()
        epoch_fr2 += avg_fr2.item()
        train_preds_all.extend(preds.detach().cpu().numpy())
        train_labels_all.extend(batch_y.cpu().numpy())

    train_loss = epoch_loss / steps_per_epoch
    train_acc = epoch_correct / epoch_total
    train_f1 = f1_score(train_labels_all, train_preds_all, average='macro')
    avg_fr1 = epoch_fr1 / steps_per_epoch
    avg_fr2 = epoch_fr2 / steps_per_epoch

    scheduler.step()

    # --- Validate (using EMA weights) ---
    ema.apply(model)
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0
    val_preds_all, val_labels_all = [], []

    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            batch_x = batch_x.to(CONFIG['DEVICE'])
            batch_y = batch_y.to(CONFIG['DEVICE'])
            _, _, mem3_rec = model(batch_x)
            T, B, C = mem3_rec.shape
            mem_flat = mem3_rec.reshape(T * B, C)
            targets_flat = batch_y.unsqueeze(0).expand(T, -1).reshape(T * B)
            loss = loss_fn(mem_flat, targets_flat)
            val_loss_sum += loss.item()
            preds = mem3_rec[-1].argmax(dim=1)
            val_correct += (preds == batch_y).sum().item()
            val_total += len(batch_y)
            val_preds_all.extend(preds.cpu().numpy())
            val_labels_all.extend(batch_y.numpy())

    val_loss = val_loss_sum / len(val_loader)
    val_acc = val_correct / val_total
    val_f1 = f1_score(val_labels_all, val_preds_all, average='macro')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_val_acc = val_acc
        best_model_state = copy.deepcopy(model.state_dict())

    ema.restore(model)

    # Log
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['train_f1'].append(train_f1)
    history['val_f1'].append(val_f1)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    history['firing_rate_l1'].append(avg_fr1)
    history['firing_rate_l2'].append(avg_fr2)

    if (epoch + 1) % 5 == 0 or epoch == 0:
        elapsed = time.time() - t_start
        print(f'Ep {epoch+1:3d}/{CONFIG["EPOCHS"]} | '
              f'Loss: {train_loss:.4f}/{val_loss:.4f} | '
              f'Acc: {train_acc:.3f}/{val_acc:.3f} | '
              f'F1: {train_f1:.3f}/{val_f1:.3f} | '
              f'FR: {avg_fr1:.2f}/{avg_fr2:.2f} | '
              f'LR: {optimizer.param_groups[0]["lr"]:.1e} | '
              f'{elapsed:.0f}s')

total_time = time.time() - t_start
print(f'\nTraining complete in {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'Best val F1: {best_val_f1:.4f} (acc: {best_val_acc:.4f})')

In [ ]:
# Cell 12: Training Curves
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0, 0].plot(history['train_loss'], label='Train')
axes[0, 0].plot(history['val_loss'], label='Val')
axes[0, 0].set_title('Loss')
axes[0, 0].legend()
axes[0, 0].set_xlabel('Epoch')

axes[0, 1].plot(history['train_acc'], label='Train')
axes[0, 1].plot(history['val_acc'], label='Val')
axes[0, 1].set_title('Accuracy')
axes[0, 1].legend()
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylim(0, 1.05)

axes[0, 2].plot(history['train_f1'], label='Train')
axes[0, 2].plot(history['val_f1'], label='Val')
axes[0, 2].set_title('Macro F1 Score')
axes[0, 2].legend()
axes[0, 2].set_xlabel('Epoch')
axes[0, 2].set_ylim(0, 1.05)

axes[1, 0].plot(history['lr'])
axes[1, 0].set_title('Learning Rate (Cosine)')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_yscale('log')

axes[1, 1].plot(history['firing_rate_l1'], label='Layer 1 (128 neurons)')
axes[1, 1].plot(history['firing_rate_l2'], label='Layer 2 (64 neurons)')
axes[1, 1].axhline(y=CONFIG['SPIKE_REG_TARGET'], color='r', linestyle='--', alpha=0.5, label=f'Target ({CONFIG["SPIKE_REG_TARGET"]})')
axes[1, 1].set_title('Avg Firing Rate')
axes[1, 1].legend()
axes[1, 1].set_xlabel('Epoch')

axes[1, 2].plot(np.array(history['train_loss']) - np.array(history['val_loss']), color='purple')
axes[1, 2].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
axes[1, 2].set_title('Train-Val Loss Gap (overfitting monitor)')
axes[1, 2].set_xlabel('Epoch')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 13: Evaluation
model.load_state_dict(best_model_state)
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch_x, batch_y in val_loader:
        _, _, mem3_rec = model(batch_x.to(CONFIG['DEVICE']))
        preds = mem3_rec[-1].argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch_y.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(8, 6))
disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
disp.plot(ax=ax, cmap='Blues')
ax.set_title(f'Confusion Matrix (Val Acc: {best_val_acc:.4f})')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 14: Feature Importance
w1 = model.fc1.weight.data.cpu().numpy()
importance = np.abs(w1).sum(axis=0)
importance = importance / importance.max()

sorted_idx = np.argsort(importance)[::-1]

fig, ax = plt.subplots(figsize=(14, 6))
ax.barh(range(len(importance)), importance[sorted_idx], color='steelblue')
ax.set_yticks(range(len(importance)))
ax.set_yticklabels([FEATURE_NAMES[i] for i in sorted_idx], fontsize=8)
ax.set_xlabel('Relative Importance (L1 norm)')
ax.set_title('Feature Importance from First Layer Weights')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Top 10 features:')
for rank, idx in enumerate(sorted_idx[:10]):
    print(f'  {rank+1}. {FEATURE_NAMES[idx]}: {importance[idx]:.4f}')

In [ ]:
# Cell 15: Hysteresis Simulation
def run_hysteresis(probs, entry_thresh, exit_thresh):
    """Apply per-class hysteresis. Returns detection state per window."""
    n_windows, n_classes = probs.shape
    detections = np.zeros(n_windows, dtype=np.int64) + 2  # default: nothing
    active_class = -1
    
    for i in range(n_windows):
        if active_class == -1:
            # Not in detection - check entry for human (0) and car (1)
            for c in [0, 1]:
                if probs[i, c] >= entry_thresh:
                    active_class = c
                    break
        else:
            # In detection - check exit
            if probs[i, active_class] < exit_thresh:
                active_class = -1
        
        if active_class >= 0:
            detections[i] = active_class
    
    return detections

def get_sequence_predictions(model, windows, labels, batch_size=64):
    """Run inference on sequential windows, return softmax probabilities."""
    model.eval()
    all_probs = []
    features_tensor = torch.tensor(
        np.clip((np.array([extract_features(w) for w in windows]) - train_mean) / train_std,
                -CONFIG['CLIP_RANGE'], CONFIG['CLIP_RANGE']),
        dtype=torch.float32
    )
    
    with torch.no_grad():
        for i in range(0, len(features_tensor), batch_size):
            batch = features_tensor[i:i+batch_size].to(CONFIG['DEVICE'])
            _, _, mem3_rec = model(batch)
            probs = torch.softmax(mem3_rec[-1], dim=1)
            all_probs.append(probs.cpu().numpy())
    
    return np.concatenate(all_probs, axis=0)

# Build sequential val windows for the human recording
human_val = split_data['human']['val']
human_val_windows = create_windows(human_val, CONFIG['WINDOW_SIZE'], CONFIG['STRIDE'])
print(f'Human val sequence: {len(human_val_windows)} windows ({len(human_val)/CONFIG["SAMPLE_RATE"]:.1f}s)')

probs = get_sequence_predictions(model, human_val_windows, None)
detections = run_hysteresis(probs, CONFIG['HYSTERESIS_ENTRY'], CONFIG['HYSTERESIS_EXIT'])

# Visualize a 30-second segment
seg_windows = 60  # 60 windows * 0.5s stride = 30s
start_w = 0
end_w = min(start_w + seg_windows, len(human_val_windows))

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

# Raw signal
t_start_s = start_w * CONFIG['STRIDE'] / CONFIG['SAMPLE_RATE']
t_end_s = (end_w * CONFIG['STRIDE'] + CONFIG['WINDOW_SIZE']) / CONFIG['SAMPLE_RATE']
sig_start = start_w * CONFIG['STRIDE']
sig_end = min(end_w * CONFIG['STRIDE'] + CONFIG['WINDOW_SIZE'], len(human_val))
t_axis = np.arange(sig_start, sig_end) / CONFIG['SAMPLE_RATE'] + len(split_data['human']['train']) / CONFIG['SAMPLE_RATE']
ax1.plot(t_axis, human_val[sig_start:sig_end], linewidth=0.3, color='black')
ax1.set_ylabel('Amplitude')
ax1.set_title('Raw Geophone Signal (Human Val Segment)')

# Softmax probabilities
window_times = np.arange(start_w, end_w) * CONFIG['STRIDE'] / CONFIG['SAMPLE_RATE'] + len(split_data['human']['train']) / CONFIG['SAMPLE_RATE']
for c, name, color in zip([0, 1, 2], CLASS_NAMES, ['blue', 'red', 'gray']):
    ax2.plot(window_times, probs[start_w:end_w, c], label=name, color=color, alpha=0.8)
ax2.axhline(y=CONFIG['HYSTERESIS_ENTRY'], color='green', linestyle='--', alpha=0.5, label=f'Entry ({CONFIG["HYSTERESIS_ENTRY"]})')
ax2.axhline(y=CONFIG['HYSTERESIS_EXIT'], color='orange', linestyle='--', alpha=0.5, label=f'Exit ({CONFIG["HYSTERESIS_EXIT"]})')
ax2.set_ylabel('Probability')
ax2.set_title('SNN Output (Softmax)')
ax2.legend(loc='upper right', fontsize=8)

# Detection overlay
colors_map = {0: 'blue', 1: 'red', 2: 'lightgray'}
for i in range(start_w, end_w):
    t = i * CONFIG['STRIDE'] / CONFIG['SAMPLE_RATE'] + len(split_data['human']['train']) / CONFIG['SAMPLE_RATE']
    ax3.axvspan(t, t + CONFIG['STRIDE']/CONFIG['SAMPLE_RATE'],
               alpha=0.6, color=colors_map[detections[i]])
ax3.set_ylabel('Detection')
ax3.set_xlabel('Time (s)')
ax3.set_title('Hysteresis Detection Output')
ax3.set_yticks([])

plt.tight_layout()
plt.show()

In [ ]:
# Cell 16: Detection Latency Analysis
def measure_detection_latency(raw_signal, probs, detections, expected_class, rms_threshold):
    """Measure delay between ground-truth event start and hysteresis trigger."""
    ws = CONFIG['WINDOW_SIZE']
    stride = CONFIG['STRIDE']
    fs = CONFIG['SAMPLE_RATE']
    
    # Find ground-truth active regions using RMS energy
    n_windows = len(probs)
    gt_active = np.zeros(n_windows, dtype=bool)
    for i in range(n_windows):
        start = i * stride
        end = start + ws
        if end <= len(raw_signal):
            rms = np.sqrt(np.mean(raw_signal[start:end]**2))
            gt_active[i] = rms >= rms_threshold
    
    # Find event starts (transition from inactive to active)
    gt_diff = np.diff(gt_active.astype(int))
    event_starts = np.where(gt_diff == 1)[0] + 1
    
    latencies = []
    for es in event_starts:
        # Find first detection after event start
        for j in range(es, min(es + 20, n_windows)):  # search up to 10s ahead
            if detections[j] == expected_class:
                latency_s = (j - es) * stride / fs + ws / fs  # include window fill time
                latencies.append(latency_s)
                break
    
    return np.array(latencies), event_starts

# Measure on human val sequence
nothing_rms = np.sqrt(np.mean(split_data['human_nothing']['val']**2))
rms_thresh = nothing_rms * 2  # activity = 2x background

latencies, events = measure_detection_latency(
    human_val, probs, detections, expected_class=0, rms_threshold=rms_thresh
)

if len(latencies) > 0:
    print(f'Detection Latency (Human, {len(latencies)} events detected / {len(events)} total):')
    print(f'  Mean:   {np.mean(latencies):.2f}s')
    print(f'  Median: {np.median(latencies):.2f}s')
    print(f'  Min:    {np.min(latencies):.2f}s')
    print(f'  Max:    {np.max(latencies):.2f}s')
    print(f'  Std:    {np.std(latencies):.2f}s')
    
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.hist(latencies, bins=20, edgecolor='black', alpha=0.7, color='steelblue')
    ax.axvline(np.mean(latencies), color='red', linestyle='--', label=f'Mean: {np.mean(latencies):.2f}s')
    ax.set_xlabel('Detection Latency (seconds)')
    ax.set_ylabel('Count')
    ax.set_title('Detection Latency Distribution')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print(f'No detections found. {len(events)} ground-truth events exist.')
    print('Consider lowering HYSTERESIS_ENTRY threshold.')

In [ ]:
# Cell 17: Summary & Export
print('=' * 60)
print('GEOPHONE SNN CLASSIFIER - FINAL SUMMARY')
print('=' * 60)
print(f'\nArchitecture: Input({CONFIG["NUM_FEATURES"]}) -> PLIF(128) -> PLIF(64) -> PLIF(3)')
print(f'Framework: SpikingJelly (ParametricLIFNode)')
print(f'Parameters: {total_params:,}')
print(f'Time steps: {CONFIG["TIME_STEPS"]}')
print(f'Spiking neurons: ~195')
print(f'\nTraining:')
print(f'  Epochs: {CONFIG["EPOCHS"]}')
print(f'  Training time: {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'  Best val F1 (macro): {best_val_f1:.4f}')
print(f'  Best val accuracy: {best_val_acc:.4f}')
print(f'  Final firing rates: L1={history["firing_rate_l1"][-1]:.3f}, L2={history["firing_rate_l2"][-1]:.3f}')
print(f'\nDetection:')
print(f'  Hysteresis: entry={CONFIG["HYSTERESIS_ENTRY"]}, exit={CONFIG["HYSTERESIS_EXIT"]}')
if len(latencies) > 0:
    print(f'  Avg detection latency: {np.mean(latencies):.2f}s')
print(f'  Theoretical latency: 1.0-1.5s + ~50ms processing')
print(f'\nData:')
print(f'  Window: {CONFIG["WINDOW_SIZE"]/CONFIG["SAMPLE_RATE"]}s, stride: {CONFIG["STRIDE"]/CONFIG["SAMPLE_RATE"]}s')
print(f'  Features: {CONFIG["NUM_FEATURES"]}')
print(f'  Train samples: {len(train_all_labels)} (incl. {len(splice_labels)} spliced)')
print(f'  Val samples: {len(val_labels)}')

# Save model and scaler
save_dir = os.path.dirname(CONFIG['DB_PATH'])
save_path = os.path.join(save_dir, 'geophone_snn_model.pt')
torch.save({
    'model_state_dict': best_model_state,
    'config': CONFIG,
    'train_mean': train_mean,
    'train_std': train_std,
    'feature_names': FEATURE_NAMES,
    'class_names': CLASS_NAMES,
    'best_val_f1': best_val_f1,
    'best_val_acc': best_val_acc,
    'history': history,
}, save_path)
print(f'\nModel saved to: {save_path}')